# AI AINAK — Notebook 2: Environmental Audio Module

**Branch 2:** Audio Ingestion, 32 kHz Preprocessing, PANNs CNN14 Bottleneck Feature Extraction, Downstream AudioMLP ESC-50 Classification, and Silence Handling.

This notebook executes the Environmental Audio branch of AI AINAK:
1. Loads the pretrained **PANNs CNN14** acoustic backbone (`Cnn14_mAP=0.431.pth`).
2. Loads the fine-tuned downstream **AudioMLP** classifier (`downstream_mlp.pt`).
3. Loads audio from `datasets/recordings/session-multimodal-01/audio.wav`.
4. Resamples to 32 kHz monaural PCM waveform.
5. Rejects zero-energy audio windows proactively (`RMS < 1e-5`).
6. Extracts a dense **2,048-dimensional bottleneck embedding** from CNN14.
7. Classifies sound classes across the 50-category ESC-50 taxonomy with softmax probabilities.
8. Saves formatted outputs to `audio_outputs.npz` for multimodal Transformer fusion.


In [ ]:
# Step 1: Install Dependencies
!pip install -q panns-inference soundfile librosa torch numpy matplotlib


In [ ]:
# Environment Detection & Path Resolution
import os, sys, hashlib
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running in Google Colab environment.")
    from google.colab import drive
    # Mount Google Drive if available
    try:
        drive.mount("/content/drive")
    except Exception as e:
        print(f"Drive mount notice: {e}")
    
    # Locate project root
    drive_candidates = [
        Path("/content/drive/MyDrive/AINAK_Colab"),
        Path("/content/drive/MyDrive/AINAK"),
        Path("/content/AINAK"),
        Path("/content"),
    ]
    PROJECT_ROOT = next((p for p in drive_candidates if (p / "artifacts").is_dir() or (p / "checkpoints").is_dir()), Path.cwd())
else:
    print("Running in local environment.")
    PROJECT_ROOT = Path.cwd().resolve()
    while not (PROJECT_ROOT / "artifacts").is_dir() and PROJECT_ROOT.parent != PROJECT_ROOT:
        PROJECT_ROOT = PROJECT_ROOT.parent

print(f"Project root resolved to: {PROJECT_ROOT}")

# Helper to find checkpoint across standard folders
def find_checkpoint(filename: str, subfolder: str = "") -> Path:
    candidates = [
        PROJECT_ROOT / "artifacts" / subfolder / filename,
        PROJECT_ROOT / "artifacts" / filename,
        PROJECT_ROOT / "checkpoints" / subfolder / filename,
        PROJECT_ROOT / "checkpoints" / filename,
        PROJECT_ROOT / filename,
        Path(filename),
    ]
    for c in candidates:
        if c.is_file():
            return c.resolve()
    raise FileNotFoundError(f"Checkpoint not found: {filename}. Checked: {[str(c) for c in candidates]}")

# Helper to find dataset / recording session
def find_session_media(session_id: str, filename: str) -> Path:
    candidates = [
        PROJECT_ROOT / "datasets" / "recordings" / session_id / filename,
        PROJECT_ROOT / "datasets" / session_id / filename,
        PROJECT_ROOT / session_id / filename,
        Path(filename),
    ]
    for c in candidates:
        if c.is_file():
            return c.resolve()
    raise FileNotFoundError(f"Media file {filename} for session {session_id} not found.")

# Ensure output directories exist
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)



## Checkpoint & Audio Ingestion
Locate CNN14, downstream MLP, and the multimodal session audio file.


In [ ]:
import soundfile as sf
import librosa
import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt

cnn14_path = find_checkpoint("Cnn14_mAP=0.431.pth", "audio")
mlp_path = find_checkpoint("downstream_mlp.pt", "audio")
audio_path = find_session_media("session-multimodal-01", "audio.wav")

print(f"Loaded PANNs CNN14 backbone: {cnn14_path} ({cnn14_path.stat().st_size / 1e6:.2f} MB)")
print(f"Loaded Downstream AudioMLP: {mlp_path} ({mlp_path.stat().st_size / 1e6:.2f} MB)")
print(f"Loaded Session Audio: {audio_path} ({audio_path.stat().st_size / 1e3:.2f} KB)")


## Downstream AudioMLP Architecture Definition
Matches `ainak.models.audio.downstream_classifier.AudioMLP`.


In [ ]:
class AudioMLP(nn.Module):
    def __init__(self, input_dim: int = 2048, hidden_dim: int = 256, num_classes: int = 50, dropout: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

# Load trained AudioMLP weights
mlp_checkpoint = torch.load(str(mlp_path), map_location="cpu", weights_only=False)
esc50_classes = mlp_checkpoint["classes"]
audio_mlp = AudioMLP(input_dim=2048, hidden_dim=256, num_classes=len(esc50_classes))
audio_mlp.load_state_dict(mlp_checkpoint["state_dict"])
audio_mlp.eval()

print(f"AudioMLP initialized successfully with {len(esc50_classes)} classes.")
print(f"Sample classes: {esc50_classes[:8]}")


## Audio Loading, Preprocessing & Sliding Windows
Loads the recording at 32 kHz and computes 2.0s windows with 1.0s stride.


In [ ]:
# Load audio waveform
waveform, sr = sf.read(str(audio_path), dtype="float32")
if waveform.ndim > 1:
    waveform = waveform.mean(axis=1) # Mono conversion

# Explicit 32 kHz resample
if sr != 32000:
    waveform = librosa.resample(waveform, orig_sr=sr, target_sr=32000)
    sr = 32000

duration = len(waveform) / sr
print(f"Audio loaded: {len(waveform)} samples @ {sr} Hz ({duration:.2f} seconds)")

# Sliding window parameters
window_sec = 2.0
hop_sec = 1.0
window_samples = int(window_sec * sr)
hop_samples = int(hop_sec * sr)

windows = []
window_timestamps = []

for start in range(0, max(1, len(waveform) - window_samples + 1), hop_samples):
    chunk = waveform[start : start + window_samples]
    if len(chunk) < window_samples:
        chunk = np.pad(chunk, (0, window_samples - len(chunk)))
    windows.append(chunk)
    window_timestamps.append(start / sr)

print(f"Extracted {len(windows)} temporal audio windows.")


## PANNs CNN14 Inference & Downstream Classification
Extracts 2,048-D bottleneck representation and runs AudioMLP classification.


In [ ]:
from panns_inference import AudioTagging
import time

# Initialize PANNs
tagger = AudioTagging(checkpoint_path=str(cnn14_path), device="cpu")

timestamps = []
embeddings = []
predicted_classes = []
confidences = []
is_silent_list = []
latencies_ms = []

print("Running CNN14 + AudioMLP inference on audio windows...")
for idx, (ts, chunk) in enumerate(zip(window_timestamps, windows)):
    t_start = time.perf_counter()
    
    # RMS silence check
    rms = float(np.sqrt(np.mean(chunk * chunk)))
    if rms < 1e-5:
        # Silence window
        emb_2048 = np.zeros(2048, dtype=np.float32)
        top_cls = "silence"
        top_conf = 0.0
        is_sil = True
    else:
        # Forward pass CNN14
        clipwise, emb = tagger.inference(chunk[None, :])
        emb_2048 = np.asarray(emb).reshape(-1).astype(np.float32)
        
        # Forward pass downstream MLP
        with torch.no_grad():
            x_tensor = torch.from_numpy(emb_2048[None, :])
            logits = audio_mlp(x_tensor)
            probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
        
        top_idx = int(probs.argmax())
        top_cls = esc50_classes[top_idx]
        top_conf = float(probs[top_idx])
        is_sil = False

    t_latency = (time.perf_counter() - t_start) * 1000

    timestamps.append(ts)
    embeddings.append(emb_2048)
    predicted_classes.append(top_cls)
    confidences.append(top_conf)
    is_silent_list.append(is_sil)
    latencies_ms.append(t_latency)

    print(f"Window {idx+1} @ {ts:.2f}s: Class='{top_cls}' (Conf={top_conf*100:.1f}%) | Latency={t_latency:.1f}ms")

print(f"\nMean Audio Latency: {np.mean(latencies_ms):.2f} ms")


## Waveform & Spectrogram Visualization


In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))

# Waveform
t_axis = np.linspace(0, duration, len(waveform))
ax1.plot(t_axis, waveform, color="#f59e0b", lw=0.8)
ax1.set_title("Environmental Audio Waveform (32 kHz)", fontsize=11, fontweight="bold")
ax1.set_xlabel("Time (seconds)")
ax1.set_ylabel("Amplitude")
ax1.grid(True, alpha=0.3)

# Spectrogram
S = librosa.feature.melspectrogram(y=waveform, sr=sr, n_mels=64)
S_dB = librosa.power_to_db(S, ref=np.max)
img = librosa.display.specshow(S_dB, x_axis="time", y_axis="mel", sr=sr, ax=ax2, cmap="magma")
fig.colorbar(img, ax=ax2, format="%+2.0f dB")
ax2.set_title("Log-Mel Spectrogram", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()


## Export Audio Outputs to `audio_outputs.npz`


In [ ]:
audio_out_path = OUTPUTS_DIR / "audio_outputs.npz"

np.savez_compressed(
    str(audio_out_path),
    timestamps=np.array(timestamps, dtype=np.float64),
    features=np.array(embeddings, dtype=np.float32),
    predicted_classes=np.array(predicted_classes),
    confidences=np.array(confidences, dtype=np.float32),
    is_silent=np.array(is_silent_list, dtype=bool),
    latencies_ms=np.array(latencies_ms, dtype=np.float32)
)

print(f"Successfully exported audio outputs to: {audio_out_path}")
print(f"File size: {audio_out_path.stat().st_size / 1e3:.1f} KB")
print(f"Feature matrix shape: {np.array(embeddings).shape} (Matches 2,048-D Transformer contract)")
